## 边构造

In [1]:
import os
import getpass

import numpy as np
import pandas as pd
import torch
import tushare as ts
from torch_geometric.data import HeteroData

In [ ]:
# 单元 1.5/5 — 公共配置与工具
# 说明：把后续各单元会用到的路径、参数和加载函数先统一定义好
DATA_DIR = 'data'  # 数据目录
STATIC_TOP_K = 20  # 静态边每个节点保留的邻居上限
CORR_TOP_K = 20  # 相关性边每个节点保留的邻居上限
CORR_LOOKBACK = 20  # 相关性计算回看窗口

# 比喻注释：这里相当于先把厨房的调料台摆好，后面做菜时就不用每次临时找
# 运行时获取 Tushare token（优先环境变量，否则提示输入）
def get_token_manual_or_env():
    """获取 Tushare token（优先环境变量，否则隐藏式输入）。"""
    tushare_token = os.environ.get('TUSHARE_TOKEN')
    if not tushare_token:
        tushare_token = getpass.getpass('请输入你的 TUSHARE_TOKEN（输入时隐藏）：').strip()
    if not tushare_token:
        raise RuntimeError('未提供 TUSHARE_TOKEN')
    print('✅ 已获取 TUSHARE_TOKEN')
    return tushare_token


def read_first_existing_csv(candidates):
    for path in candidates:
        if os.path.exists(path):
            print(f'✅ 找到文件：{path}')
            return pd.read_csv(path)
    print('警告：未找到候选 CSV 文件：' + ', '.join(candidates))
    return None


def load_stock_universe(data_dir=DATA_DIR):
    """读取 stock_pool_all.csv 与 node_feats_allx10.npy，并返回 stock_codes, node_feats"""
    pool_path = os.path.join(data_dir, 'stock_pool_all.csv')
    feat_path = os.path.join(data_dir, 'node_feats_allx10.npy')
    if not os.path.exists(pool_path) or not os.path.exists(feat_path):
        raise FileNotFoundError('缺少 stock_pool_all.csv 或 node_feats_allx10.npy')
    pool_df = pd.read_csv(pool_path)
    feat_array = np.load(feat_path, allow_pickle=True)
    if 'ts_code' not in pool_df.columns:
        raise ValueError('stock_pool_all.csv 中缺少 ts_code 列')
    stock_codes = pool_df['ts_code'].astype(str).tolist()
    if len(stock_codes) != feat_array.shape[0]:
        raise ValueError(f'代码索引表与特征矩阵行数不一致：{len(stock_codes)} != {feat_array.shape[0]}')
    node_feats = np.asarray(feat_array, dtype=np.float32)
    print(f'✅ 已按 code 索引表加载节点特征：{node_feats.shape}')
    return stock_codes, node_feats


def build_stock2idx(stock_codes):
    return {code: idx for idx, code in enumerate(stock_codes)}


def edge_index_from_pairs(edge_pairs):
    """把 (src,dst) 对集合转换为 PyG 的 edge_index 张量。"""
    if not edge_pairs:
        return torch.empty((2, 0), dtype=torch.long)
    pair_tensor = torch.tensor(sorted(edge_pairs), dtype=torch.long)
    return pair_tensor.t().contiguous()


In [ ]:
# 抓取日线行情
# 说明：按 `stock_codes` 列表逐支拉取收盘价并计算日收益率，支持从已有文件恢复以节省重复下载。
import time

DATA_DIR = globals().get('DATA_DIR', 'data')  # 数据目录（若前置单元未定义则使用默认值）
if 'get_token_manual_or_env' not in globals():
    def get_token_manual_or_env():
        """获取 Tushare token（优先环境变量，否则隐藏式输入）。"""
        tushare_token = os.environ.get('TUSHARE_TOKEN')
        if not tushare_token:
            tushare_token = getpass.getpass('请输入你的 TUSHARE_TOKEN（输入时隐藏）：').strip()
        if not tushare_token:
            raise RuntimeError('未提供 TUSHARE_TOKEN')
        print('✅ 已获取 TUSHARE_TOKEN')
        return tushare_token

def fetch_daily_returns_for_codes(stock_codes, lookback_days=252, end_date=None, data_directory=DATA_DIR, pause_seconds=0.4, checkpoint_every=500):
    """为给定 stock_codes 拉取最近窗口的收盘价并计算日收益率，结果保存为 data/daily_returns.csv。

    支持从已有的 daily_returns.csv 恢复（避免重复下载），返回升序索引的 DataFrame（trade_date）。
    """
    # 运行时获取 Tushare token（优先环境变量，否则提示输入）
    tushare_token = get_token_manual_or_env()
    pro_api = ts.pro_api(tushare_token)  # Tushare API 客户端

    end_date_str = end_date or pd.Timestamp.today().strftime('%Y%m%d')  # 结束日期（字符串）
    start_date_str = (pd.Timestamp(end_date_str) - pd.Timedelta(days=int(lookback_days * 2))).strftime('%Y%m%d')  # 开始日期（多取以防停牌）

    os.makedirs(data_directory, exist_ok=True)
    output_csv_path = os.path.join(data_directory, 'daily_returns.csv')  # 输出文件路径

    # 若已存在结果文件，则读取以支持断点续传
    existing_dataframe = None
    if os.path.exists(output_csv_path):
        try:
            existing_dataframe = pd.read_csv(output_csv_path, index_col=0, parse_dates=True)
            print(f'✅ 发现已有收益率文件，已加载（可续传），shape={existing_dataframe.shape}')
        except Exception as e:
            print('错误：读取已有收益率文件失败，已忽略并从头抓取。错误详情：' + str(e))
            existing_dataframe = None

    # 已抓取的映射：股票代码 -> 收益率序列
    fetched_series_map = {} if existing_dataframe is None else {column: existing_dataframe[column].dropna() for column in existing_dataframe.columns}

    # 比喻注释：这里相当于厨师按菜谱（一支支股票）去市场买菜（拉取历史收盘价），并把菜洗净切好（计算收益率）
    for index_number, stock_code in enumerate(stock_codes):
        if stock_code in fetched_series_map and len(fetched_series_map[stock_code]) >= lookback_days:
            # 已有足够数据，跳过
            continue
        try:
            price_dataframe = pro_api.daily(ts_code=stock_code, start_date=start_date_str, end_date=end_date_str)
            if price_dataframe is None or price_dataframe.empty:
                print(f'警告：{stock_code} 未返回行情数据，已跳过')
                continue
            price_dataframe = price_dataframe.sort_values('trade_date')[['trade_date', 'close']].drop_duplicates('trade_date')
            price_dataframe['trade_date'] = pd.to_datetime(price_dataframe['trade_date'])
            price_dataframe = price_dataframe.set_index('trade_date')
            price_dataframe = price_dataframe[-lookback_days:]

            # 比喻注释：把收盘价变成收益率，相当于把原料加工成可以直接下锅的食材
            return_series = price_dataframe['close'].pct_change().fillna(0.0)
            fetched_series_map[stock_code] = return_series

            if (index_number + 1) % 200 == 0:
                print(f'✅ 已尝试处理 {index_number+1}/{len(stock_codes)} 支股票')
        except Exception as e:
            # 直白错误提示，指导下一步
            print(f'错误：抓取 {stock_code} 的行情失败。请检查网络或 token 是否有效。股票代码={stock_code}。详细错误：{e}')
        time.sleep(pause_seconds)

        # 定期保存以防中断
        if (index_number + 1) % checkpoint_every == 0:
            try:
                checkpoint_dataframe = pd.DataFrame({k: v for k, v in fetched_series_map.items()})
                checkpoint_dataframe = checkpoint_dataframe.sort_index()
                checkpoint_dataframe.to_csv(output_csv_path, index_label='trade_date')
                print(f'✅ 已 checkpoint 保存 {output_csv_path}，已处理 {index_number+1} 支')
            except Exception as e:
                print('错误：保存 checkpoint 失败，请检查磁盘权限与可用空间。详细错误：' + str(e))

    if not fetched_series_map:
        print('警告：未获取到任何行情数据，请确认 TUSHARE_TOKEN 与网络是否正常')
        return None

    try:
        result_dataframe = pd.DataFrame({k: v for k, v in fetched_series_map.items()}).sort_index()
        result_dataframe.to_csv(output_csv_path, index_label='trade_date')
        print('✅ 已保存：' + output_csv_path)
        return result_dataframe
    except Exception as e:
        print('错误：保存最终收益率文件失败。请检查磁盘或路径权限。详细错误：' + str(e))
        raise


NameError: name 'DATA_DIR' is not defined

In [ ]:
# 抓取行业映射并按顺序对齐（单元 4.2/5）
# 说明：使用 tushare 获取行业信息并按照传入的 stock_codes 顺序保存为 data/industry_mapping.csv

def fetch_industry_mapping_aligned(stock_codes, data_directory=DATA_DIR):
    # 比喻注释：这里相当于餐厅管理员核对菜品清单，把每道菜对应到菜单的分类上
    tushare_token = get_token_manual_or_env()  # Tushare 访问令牌
    pro_api = ts.pro_api(tushare_token)  # Tushare API 客户端

    industry_dataframe = pro_api.stock_basic(exchange='', list_status='L', fields='ts_code,industry')
    if industry_dataframe is None or industry_dataframe.empty:
        print('错误：从 Tushare 获取行业分类失败，返回为空。请检查 token 权限或稍后重试')
        raise RuntimeError('tushare 返回空的 stock_basic')

    industry_dataframe = industry_dataframe[industry_dataframe['ts_code'].astype(str).isin(stock_codes)].copy()
    industry_dataframe['ts_code'] = industry_dataframe['ts_code'].astype(str)

    mapping_dictionary = industry_dataframe.set_index('ts_code')['industry'].to_dict()  # 映射字典：ts_code -> industry

    output_dataframe = pd.DataFrame({'ts_code': stock_codes})  # 按传入顺序构造输出表
    output_dataframe['industry'] = output_dataframe['ts_code'].map(mapping_dictionary).fillna('UNKNOWN')

    try:
        os.makedirs(data_directory, exist_ok=True)
        output_path = os.path.join(data_directory, 'industry_mapping.csv')
        output_dataframe.to_csv(output_path, index=False)
        print('✅ 已保存：' + output_path)
        return output_dataframe
    except Exception as e:
        print('错误：保存行业映射失败，请检查磁盘权限或路径是否存在。详细错误：' + str(e))
        raise


In [ ]:
# 运行示例（单元 4.3/5）
print('提示：将按 `stock_pool_all.csv` / `node_feats_allx10.csv` 的顺序抓取并对齐。')
try:
    stock_codes, _ = load_stock_universe(DATA_DIR)
    print(f"✅ 节点列表已加载，股票数量={len(stock_codes)}")
except Exception as e:
    print('错误：未能加载节点列表，请先生成 data/stock_pool_all.csv 与 data/node_feats_allx10.csv。详细错误：' + str(e))
    raise

print(f'将为 {len(stock_codes)} 支股票抓取日线与行业分类，若数量较大请耐心等待。')
# 若需要只抓取部分用于测试，可替换为 stock_codes[:100]
# stock_codes = stock_codes[:100]

# 先抓取日线并保存 return 面板（支持断点续传）
ret_df = fetch_daily_returns_for_codes(stock_codes, lookback_days=max(60, CORR_LOOKBACK*5))
if ret_df is None:
    print('错误：未能获取到任何日线数据，请检查 TUSHARE_TOKEN 与网络连接。')
else:
    print('✅ 日线收益率面板已生成（或加载）')

# 再抓取行业并按顺序保存
ind_df = fetch_industry_mapping_aligned(stock_codes)
print('✅ 行业映射已生成（按节点顺序对齐）')

print('完成：日线与行业映射已抓取并按节点顺序保存。')


In [ ]:
# 单元 1/5 — 导入、常量与简单工具
# 说明：此单元只做导入与常量定义，便于后续按单元执行和调试
# 特征列顺序（与 main01 保持一致）
FEATURE_COLUMNS = [
    'ln_cap', 'beta', 'momentum_20d', 'volatility_60d', 'turnover_20d',
    'pe_ttm', 'pb', 'roe', 'debt_ratio', 'eps_growth'
]

# 数据与参数
DATA_DIR = 'data'
STATIC_TOP_K = 20
CORR_TOP_K = 20
CORR_LOOKBACK = 20


def get_token_manual_or_env():
    """获取 Tushare token（优先环境变量，否则隐藏式输入）。"""
    token = os.environ.get('TUSHARE_TOKEN')
    if not token:
        token = getpass.getpass('请输入你的 TUSHARE_TOKEN（输入时隐藏）：').strip()
    if not token:
        raise RuntimeError('未提供 TUSHARE_TOKEN')
    print('✅ 已获取 TUSHARE_TOKEN')
    return token


In [ ]:
# 单元 2/5 — I/O 与节点加载工具
# 说明：加载已有 nodes/features，并返回按 stock_code 对齐的矩阵与代码列表

def read_first_existing_csv(candidates):
    for path in candidates:
        if os.path.exists(path):
            print(f'✅ 找到文件：{path}')
            return pd.read_csv(path)
    print('警告：未找到候选 CSV 文件：' + ', '.join(candidates))
    return None


def load_stock_universe(data_dir=DATA_DIR):
    """读取 stock_pool_all.csv 与 node_feats_allx10.csv，并返回 stock_codes, node_feats"""
    pool_path = os.path.join(data_dir, 'stock_pool_all.csv')
    feat_path = os.path.join(data_dir, 'node_feats_allx10.csv')
    if not os.path.exists(pool_path) or not os.path.exists(feat_path):
        raise FileNotFoundError('缺少 stock_pool_all.csv 或 node_feats_allx10.csv')
    pool_df = pd.read_csv(pool_path)
    feat_df = pd.read_csv(feat_path, index_col=0)
    if 'ts_code' not in pool_df.columns:
        raise ValueError('stock_pool_all.csv 中缺少 ts_code 列')
    stock_codes = pool_df['ts_code'].astype(str).tolist()
    feat_df = feat_df.reindex(stock_codes)
    node_feats = feat_df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=np.float32)
    print(f'✅ 已加载节点特征：{node_feats.shape}')
    return stock_codes, node_feats


def build_stock2idx(stock_codes):
    return {code: idx for idx, code in enumerate(stock_codes)}


def edge_index_from_pairs(edge_pairs):
    """把 (src,dst) 对集合转换为 PyG 的 edge_index 张量。"""
    if not edge_pairs:
        return torch.empty((2, 0), dtype=torch.long)
    pair_tensor = torch.tensor(sorted(edge_pairs), dtype=torch.long)
    return pair_tensor.t().contiguous()


In [ ]:
# 单元 3/5 — 静态边（行业/概念）构造函数
# 说明：将行业/概念映射转换为节点对，并限制每节点邻居数

def build_group_edge_index(mapping_df, stock2idx, stock_col, group_col, max_neighbors=STATIC_TOP_K):
    valid = mapping_df[[stock_col, group_col]].dropna().copy()
    valid[stock_col] = valid[stock_col].astype(str)
    valid[group_col] = valid[group_col].astype(str)
    grouped = valid.groupby(group_col)[stock_col].apply(lambda series: sorted(set(series))).to_dict()
    edge_pairs = set()
    clipped_groups = 0
    for members in grouped.values():
        members = [code for code in members if code in stock2idx]
        if len(members) < 2:
            continue
        for src_code in members:
            peers = [code for code in members if code != src_code]
            if len(peers) > max_neighbors:
                peers = peers[:max_neighbors]
                clipped_groups += 1
            src_idx = stock2idx[src_code]
            for dst_code in peers:
                dst_idx = stock2idx[dst_code]
                edge_pairs.add((src_idx, dst_idx))
                edge_pairs.add((dst_idx, src_idx))
    edge_index = edge_index_from_pairs(edge_pairs)
    print(f'✅ {group_col} 边构造完成：{edge_index.shape[1]} 条，裁剪次数={clipped_groups}')
    return edge_index


def load_industry_mapping(stock_codes):
    """使用 tushare 拉取申万行业分类并过滤到我们的股票池。"""
    token = get_token_manual_or_env()
    pro = ts.pro_api(token)
    df = pro.stock_basic(exchange='', list_status='L', fields='ts_code,industry')
    df = df[df['ts_code'].astype(str).isin(stock_codes)].copy()
    df['ts_code'] = df['ts_code'].astype(str)
    df['industry'] = df['industry'].fillna('UNKNOWN').astype(str)
    print(f'✅ 行业分类加载完成：{df["industry"].nunique()} 个行业')
    return df[['ts_code', 'industry']]


def load_concept_mapping(data_dir=DATA_DIR):
    """从本地 CSV 加载概念映射（若存在）。"""
    candidates = [
        os.path.join(data_dir, 'stock_concept.csv'),
        os.path.join(data_dir, 'concept_mapping.csv'),
        os.path.join(data_dir, 'stock_concepts.csv'),
    ]
    df = read_first_existing_csv(candidates)
    if df is None or df.empty:
        return pd.DataFrame(columns=['ts_code', 'concept'])
    if 'ts_code' not in df.columns:
        raise ValueError('概念文件中缺少 ts_code 列')
    concept_cols = [col for col in df.columns if col != 'ts_code']
    if not concept_cols:
        return pd.DataFrame(columns=['ts_code', 'concept'])
    concept_col = concept_cols[0]
    out = df[['ts_code', concept_col]].copy()
    out.columns = ['ts_code', 'concept']
    out['ts_code'] = out['ts_code'].astype(str)
    out['concept'] = out['concept'].astype(str)
    print(f'✅ 概念映射加载完成：{out["concept"].nunique()} 个概念')
    return out


In [ ]:
# 单元 4/5 — 收益率面板与 high_corr 边构造
# 说明：从本地加载日收益率面板并计算 Top-K 相关度邻居

def load_return_panel(data_dir=DATA_DIR):
    candidates = [
        os.path.join(data_dir, 'daily_returns.csv'),
        os.path.join(data_dir, 'stock_returns.csv'),
        os.path.join(data_dir, 'return_panel.csv'),
    ]
    df = read_first_existing_csv(candidates)
    if df is None or df.empty:
        return None
    if 'trade_date' in df.columns:
        df = df.sort_values('trade_date').set_index('trade_date')
    print(f'✅ 收益率面板加载完成：{df.shape}')
    return df


def normalize_return_panel(return_df, stock_codes, lookback=CORR_LOOKBACK):
    panel = return_df.copy()
    panel = panel.reindex(columns=[c for c in stock_codes if c in panel.columns])
    panel = panel.tail(lookback)
    panel = panel.apply(pd.to_numeric, errors='coerce')
    print(f'✅ 相关性窗口已截取：{panel.shape}')
    return panel


def build_high_corr_edge_index(return_df, stock_codes, top_k=CORR_TOP_K, lookback=CORR_LOOKBACK, chunk_size=256):
    if return_df is None or return_df.empty:
        print('警告：没有收益率面板，high_corr 边为空')
        return torch.empty((2, 0), dtype=torch.long), torch.empty((0,), dtype=torch.float32)
    panel = normalize_return_panel(return_df, stock_codes, lookback=lookback)
    if panel.shape[0] < 2 or panel.shape[1] < 2:
        print('警告：收益率面板有效维度不足，high_corr 边为空')
        return torch.empty((2, 0), dtype=torch.long), torch.empty((0,), dtype=torch.float32)
    values = panel.to_numpy(dtype=np.float32)
    mean = np.nanmean(values, axis=0, keepdims=True)
    std = np.nanstd(values, axis=0, keepdims=True)
    std[std == 0] = np.nan
    z_scores = np.nan_to_num((values - mean) / std, nan=0.0, posinf=0.0, neginf=0.0)
    divisor = max(z_scores.shape[0], 1)
    edge_weight_map = {}
    n_nodes = z_scores.shape[1]
    k = min(top_k, max(n_nodes - 1, 1))
    for start in range(0, n_nodes, chunk_size):
        stop = min(start + chunk_size, n_nodes)
        block = (z_scores[:, start:stop].T @ z_scores) / divisor
        for local_idx, row in enumerate(block):
            src_idx = start + local_idx
            row[src_idx] = -np.inf
            top_idx = np.argpartition(-row, k - 1)[:k]
            top_idx = top_idx[np.argsort(-row[top_idx])]
            for dst_idx in top_idx:
                if dst_idx == src_idx:
                    continue
                weight = float(row[dst_idx])
                edge_weight_map[(src_idx, dst_idx)] = max(edge_weight_map.get((src_idx, dst_idx), -np.inf), weight)
                edge_weight_map[(dst_idx, src_idx)] = max(edge_weight_map.get((dst_idx, src_idx), -np.inf), weight)
    if not edge_weight_map:
        print('警告：未构造出 high_corr 边')
        return torch.empty((2, 0), dtype=torch.long), torch.empty((0,), dtype=torch.float32)
    edge_pairs = list(edge_weight_map.keys())
    edge_weights = [edge_weight_map[pair] for pair in edge_pairs]
    edge_index = edge_index_from_pairs(edge_pairs)
    edge_weight = torch.tensor(edge_weights, dtype=torch.float32)
    print(f'✅ high_corr 边构造完成：{edge_index.shape[1]} 条')
    return edge_index, edge_weight


In [ ]:
# 单元 5/5 — 组装 HeteroData 并运行构造流程
# 说明：最终组装异构图并可选择保存为文件，便于后续训练

def build_hetero_graph(data_dir=DATA_DIR, save_path=None):
    stock_codes, node_feats = load_stock_universe(data_dir)
    stock2idx = build_stock2idx(stock_codes)
    industry_df = load_industry_mapping(stock_codes)
    concept_df = load_concept_mapping(data_dir)
    return_df = load_return_panel(data_dir)

    industry_edge_index = build_group_edge_index(industry_df, stock2idx, 'ts_code', 'industry', STATIC_TOP_K)
    concept_edge_index = build_group_edge_index(concept_df, stock2idx, 'ts_code', 'concept', STATIC_TOP_K)
    corr_edge_index, corr_edge_weight = build_high_corr_edge_index(return_df, stock_codes, CORR_TOP_K, CORR_LOOKBACK)

    graph = HeteroData()
    graph['stock'].x = torch.tensor(node_feats, dtype=torch.float32)
    graph['stock', 'same_industry', 'stock'].edge_index = industry_edge_index
    graph['stock', 'same_concept', 'stock'].edge_index = concept_edge_index
    graph['stock', 'high_corr', 'stock'].edge_index = corr_edge_index
    graph['stock', 'high_corr', 'stock'].edge_weight = corr_edge_weight

    print('✅ 异构图边构造完成')
    print(f'   same_industry: {industry_edge_index.shape[1]}')
    print(f'   same_concept: {concept_edge_index.shape[1]}')
    print(f'   high_corr: {corr_edge_index.shape[1]}')

    if save_path:
        torch.save({'graph': graph, 'stock_codes': stock_codes, 'stock2idx': stock2idx}, save_path)
        print(f'✅ 已保存图文件：{save_path}')
    return graph, stock2idx


# 运行构造流程（按需执行）
try:
    graph, stock2idx = build_hetero_graph(data_dir=DATA_DIR, save_path=os.path.join(DATA_DIR, 'hetero_edges.pt'))
except Exception as e:
    print('错误：边构造失败，请检查输入文件和 token。')
    raise
